# Estimador TRI com provas oficiais do INEP

Este notebook usa os ZIPs oficiais do INEP como fonte de **questões, gabaritos e parâmetros A/B/C**. A ENEM.DEV serve apenas de inspiração para o formato canônico dos registros: `title`, `index`, `discipline`, `year`, `context`, `files`, `correctAlternative`, `alternativesIntroduction` e `alternatives`. Não fazemos chamadas à API nem usamos seu conteúdo como fonte do dataset.

Para 2009, a tabela `ITENS_PROVA_2009.csv` relaciona `CO_POSICAO`, `SG_AREA`, `CO_ITEM`, gabarito, cor/caderno (`TX_COR`, `CO_PROVA`) e parâmetros (`NU_PARAM_A/B/C`). Os PDFs são a fonte do enunciado e das alternativas. Este notebook extrai texto dos PDFs com PyMuPDF e monta registros no esquema acima. Elementos visuais podem exigir OCR/revisão e não devem ser tratados como texto extraído.


In [ ]:
# 1. Configuração
from pathlib import Path
import os, re, json, time, hashlib, zipfile, unicodedata
import requests, pandas as pd, numpy as np
from IPython.display import display

# Atualiza certificados-raiz no runtime Colab e usa CA bundle atual do certifi.
!pip -q install --upgrade certifi
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/estimador_tri_enem')
except ImportError:
    ROOT = Path('/content/estimador_tri_enem')
RAW, EXTRACTED, OUT = (ROOT/'raw', ROOT/'extracted', ROOT/'processed')
for p in (RAW, EXTRACTED, OUT): p.mkdir(parents=True, exist_ok=True)
YEARS = [2009]  # comece pelo ZIP já baixado; acrescente anos após validar o layout
API = 'https://api.enem.dev/v1'
INEP_PAGE = 'https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem'
S = requests.Session()
S.verify = certifi.where()
S.headers['User-Agent'] = 'EstimadorTRI/1.0 educacional'
print('Certificados CA:', certifi.where())
print('Saída:', ROOT)


## 2. Fonte e formato das questões

Não buscar questões na ENEM.DEV. A coleta parte dos ZIPs do INEP. O formato abaixo apenas espelha os campos úteis da API para manter uma estrutura uniforme, acrescentando os identificadores oficiais necessários ao pareamento: `itemCode`, `examColor`, `examCode`, `position` e parâmetros de referência.


In [ ]:
# Sem coleta remota da ENEM.DEV: a prova e os rótulos vêm do pacote INEP.
questions = pd.DataFrame()
print("A fonte das questões será os PDFs oficiais do INEP.")


## 3. Carregar ZIPs oficiais do INEP

Envie os ZIPs principais baixados manualmente. A célula de upload do próximo bloco aceita `microdados_enem_<ano>.zip` e valida o arquivo, sem descompactar a base grande de respostas dos participantes.


In [ ]:
!pip -q install pymupdf pandas scikit-learn sentence-transformers openpyxl pyarrow
import fitz  # PyMuPDF: extrai texto dos cadernos PDF
from google.colab import files

uploaded = files.upload()
zip_paths = {}
for name, content in uploaded.items():
    match = MAIN_ZIP.fullmatch(Path(name).name)
    if not match:
        print(f"Ignorado: {name} (esperado microdados_enem_<ano>.zip)")
        continue
    year = int(match.group(1))
    dest = RAW / f"microdados_enem_{year}.zip"
    if not dest.exists() or dest.stat().st_size != len(content):
        dest.write_bytes(content)
    with zipfile.ZipFile(dest) as archive:
        bad = archive.testzip()
        if bad:
            raise zipfile.BadZipFile(f"ZIP corrompido no membro {bad}")
    zip_paths[year] = dest
    print(f"ZIP validado: {dest} ({dest.stat().st_size / 1024**2:.1f} MiB)")

missing = set(YEARS) - set(zip_paths)
if missing:
    print(f"Ainda faltam os ZIPs de {sorted(missing)}; ajuste YEARS ou envie-os.")


## 4. Inspecionar ZIPs e extrair somente arquivos úteis

A base `MICRODADOS_ENEM_<ano>.csv` (respostas individuais, muito grande) não será extraída. Precisamos dos arquivos `ITENS_PROVA`, dicionário/leia-me e PDFs de provas/gabaritos. Para evitar encher a memória, extraímos esses membros seletivamente.


In [ ]:
def manifest(zpath):
    with zipfile.ZipFile(zpath) as z:
        return pd.DataFrame([{"name": i.filename, "bytes": i.file_size} for i in z.infolist() if not i.is_dir()])

def extract_useful(year, zpath):
    target = EXTRACTED / str(year)
    target.mkdir(parents=True, exist_ok=True)
    kept = []
    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            if info.is_dir():
                continue
            name = Path(info.filename.replace("\\", "/")).name
            if not name:
                continue
            upper = name.upper()
            useful = ("ITENS_PROVA" in upper or upper.endswith(".PDF") and
                      ("PROVA_DIA" in upper or "GAB_DIA" in upper or "LEIA_ME" in upper))
            if not useful:
                continue
            dest = target / name
            if not dest.exists() or dest.stat().st_size != info.file_size:
                with z.open(info) as src, open(dest, "wb") as dst:
                    while True:
                        chunk = src.read(1024 * 1024)
                        if not chunk: break
                        dst.write(chunk)
            kept.append(dest)
    return kept

manifests, files_by_year = {}, {}
for year, zpath in zip_paths.items():
    manifests[year] = manifest(zpath)
    print("Conteúdo do ZIP", year); display(manifests[year])
    files_by_year[year] = extract_useful(year, zpath)
    print("Arquivos úteis extraídos:", [p.name for p in files_by_year[year]])


## 5. Normalizar a tabela oficial de parâmetros

A tabela oficial define os rótulos reais. Cada registro final será uma questão de um caderno específico (ano + cor/código de prova + área + posição), não apenas um código de item: o mesmo item pode aparecer em posições diferentes nos cadernos.


In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    return re.sub(r"[^A-Z0-9]+", "_", s).strip("_")

parameters_by_year = {}
for year, paths in files_by_year.items():
    item_path = next((p for p in paths if "ITENS_PROVA" in p.name.upper()), None)
    if item_path is None:
        print("Tabela ITENS_PROVA não encontrada para", year); continue
    df = pd.read_csv(item_path, sep=";", encoding="latin1", dtype=str).fillna("")
    required = {"CO_POSICAO", "SG_AREA", "CO_ITEM", "TX_GABARITO", "TX_COR", "CO_PROVA",
                "NU_PARAM_A", "NU_PARAM_B", "NU_PARAM_C"}
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(f"{item_path.name}: faltam colunas {sorted(missing_cols)}")
    for col in ("NU_PARAM_A", "NU_PARAM_B", "NU_PARAM_C"):
        df[col] = pd.to_numeric(df[col].str.replace(",", ".", regex=False), errors="coerce")
    df["CO_POSICAO"] = pd.to_numeric(df["CO_POSICAO"], errors="coerce").astype('Int64')
    df["year"] = year
    parameters_by_year[year] = df
    print(year, df.shape, "cores:", sorted(df.TX_COR.unique()), "áreas:", sorted(df.SG_AREA.unique()))


In [ ]:
## 6. Construir registros de questão no esquema inspirado na ENEM.DEV

PyMuPDF extrai texto de cada PDF. Para o protótipo, cada item do caderno é representado por `context` (texto extraído da região correspondente), `alternatives` (extraídas quando reconhecíveis; se falhar, ficam vazias e o texto integral permanece em `context`) e campos canônicos. Isso permite avançar sem perder o gabarito/parâmetros oficiais. A segmentação automática de páginas/questões precisa de inspeção; os registros carregam `extractionStatus` para distinguir extração automática de revisão.


# Mapeamento de provas/cadernos 2009 -> dia, cor e códigos CO_PROVA por área.
# A tabela ITENS_PROVA é a fonte final de CO_PROVA; este mapa guia a seleção dos PDFs.
PDF_FORM_RE = re.compile(r"ENEM_(\d{4})_PROVA_DIA_(\d+)_(AZUL|AMARELO|BRANCO|ROSA|CINZA)_(\d+)", re.I)
AREA_LABEL = {"CH":"ciencias-humanas", "CN":"ciencias-natureza", "LC":"linguagens", "MT":"matematica"}


def pdf_text(path):
    doc = fitz.open(path)
    return "\n".join(page.get_text("text", sort=True) for page in doc)

def parse_question_blocks(text):
    # Segmentador conservador por marcadores QUESTÃO/QUESTAO; preservar bloco inteiro.
    starts = list(re.finditer(r"(?im)^\s*QUEST(?:Ã|A)O\s+(\d+)\b", text))
    blocks = []
    for i, m in enumerate(starts):
        end = starts[i+1].start() if i+1 < len(starts) else len(text)
        block = text[m.start():end].strip()
        blocks.append((int(m.group(1)), block))
    return blocks

def extract_alternatives(block):
    # Aceita marcadores A-E no início de linha ou após espaços; texto pode conter imagens ausentes do PDF text layer.
    matches = list(re.finditer(r"(?m)^\s*([A-E])\s*[)\].-]\s*", block))
    alts=[]
    for i,m in enumerate(matches):
        end=matches[i+1].start() if i+1<len(matches) else len(block)
        alts.append({"letter":m.group(1),"text":block[m.end():end].strip(),"file":None,"isCorrect":False})
    return alts

records=[]
for year, paths in files_by_year.items():
    param=parameters_by_year.get(year)
    if param is None: continue
    for path in paths:
        m=PDF_FORM_RE.search(path.stem)
        if not m: continue
        pdf_year, day, color, form_no = int(m.group(1)), int(m.group(2)), m.group(3).upper(), int(m.group(4))
        pdf_text_all=pdf_text(path)
        blocks=parse_question_blocks(pdf_text_all)
        # filtra linhas do INEP para este ano e cor; mantém chave oficial de prova, área e posição
        subset=param[param.TX_COR.str.upper().eq(color)]
        for area, group in subset.groupby("SG_AREA"):
            codes=sorted(group.CO_PROVA.astype(str).unique())
            # Se houver múltiplos códigos por área/cor (regular/reaplicação), só seleciona pelo nome do caderno quando inequívoco.
            if len(codes)!=1:
                print(f"Revisar códigos de prova: ano={year}, cor={color}, área={area}, códigos={codes}")
            exam_code=codes[0] if codes else None
            area_rows=group[group.CO_PROVA.astype(str).eq(exam_code)].copy() if exam_code else group.iloc[0:0]
            for _, row in area_rows.iterrows():
                pos=int(row.CO_POSICAO)
                # 2009 posições globais 46-90 / 91-135 / 136-180; número local dentro da área (1-45)
                local_index=((pos-46)%45)+1 if pos>=46 else pos
                candidate=next((b for qn,b in blocks if qn==pos), None)
                if candidate is None:
                    candidate=next((b for qn,b in blocks if qn==local_index), "")
                alts=extract_alternatives(candidate)
                correct=str(row.TX_GABARITO).strip().upper()
                for alt in alts: alt["isCorrect"]=(alt["letter"]==correct)
                discipline=AREA_LABEL.get(area,area)
                records.append({
                    "title":f"Questão {local_index} - ENEM {year}", "index":local_index,
                    "discipline":discipline,"year":year,"context":candidate,
                    "files":[],"correctAlternative":correct or None,
                    "alternativesIntroduction":None,"alternatives":alts,
                    "itemCode":str(row.CO_ITEM),"examColor":color,"examCode":exam_code,
                    "position":pos,"day":day,"difficultyA":row.NU_PARAM_A,
                    "difficultyB":row.NU_PARAM_B,"guessingC":row.NU_PARAM_C,
                    "extractionStatus":"needs_review" if not candidate or len(alts)!=5 else "auto_extracted"
                })
questions=pd.DataFrame(records)
print("Registros montados:",len(questions))
if len(questions):
    display(questions[["year","day","examColor","examCode","discipline","index","position","itemCode","correctAlternative","difficultyA","difficultyB","guessingC","extractionStatus"]].head(20))


In [ ]:
## 7. Revisão e exportação do dataset

Antes de treinar, confira blocos e alternativas contra PDFs. O dataset inclui todos os parâmetros disponíveis, mas registros com `extractionStatus=needs_review` precisam de correção/validação. Exportamos CSV/Parquet e um JSON no formato inspirado na API.


# Campos estáveis (sem DataFrame interno nem paths temporários).
if questions.empty:
    raise ValueError("Nenhuma questão foi extraída dos PDFs selecionados.")
canonical_cols=["title","index","discipline","year","context","files","correctAlternative",
                "alternativesIntroduction","alternatives","itemCode","examColor","examCode","position","day",
                "difficultyA","difficultyB","guessingC","extractionStatus"]
questions=questions[canonical_cols].copy()
questions.to_json(OUT/"questoes_inep_formato_enemdev.json",orient="records",force_ascii=False,indent=2)
questions.drop(columns=["alternatives","files"]).to_csv(OUT/"questoes_inep_indice.csv",index=False)
questions.to_parquet(OUT/"dataset_questoes_rotulado.parquet",index=False)
print("Exportados:", OUT)
print(questions.extractionStatus.value_counts(dropna=False))


In [ ]:
## 8. Próximo passo: embeddings e regressão

Aplicar embeddings em `context` (texto do enunciado e alternativas) e prever os alvos `difficultyA`, `difficultyB`, `guessingC`. Para uma avaliação válida, usar divisão por `itemCode` e comparar MAE/RMSE/R² com baseline. Só incluir após revisar os campos `context` e alternativas; os PDFs de 2009 contêm imagens/gráficos que a extração textual pode perder.


# Verificação rápida da qualidade da montagem antes do treino.
quality=(questions.groupby(["year","examColor","discipline"],dropna=False)
         .agg(n_questions=("itemCode","size"),
              n_missing_context=("context",lambda s:int(s.fillna("").str.len().eq(0).sum())),
              n_alternatives=("alternatives",lambda s:int(s.map(len).ne(5).sum())),
              n_duplicate_positions=("position",lambda s:int(s.duplicated().sum())))
         .reset_index())
display(quality)
quality.to_csv(OUT/"qualidade_extracao.csv",index=False)


In [ ]:
# Treino intencionalmente separado da extração: primeiro revisar a qualidade_extracao.csv e os PDFs.
# Depois podemos habilitar a célula de embeddings/regressão com os registros aprovados.
print("Revise os registros e confirme a extração antes do treino.")


## Fontes e limitações

A estrutura dos campos visíveis é inspirada no contrato documentado da ENEM.DEV; o conteúdo e os alvos são dos arquivos oficiais do INEP. PDFs podem incluir gráficos/imagens sem camada de texto. Em 2009, posições são ligadas ao código oficial do item e caderno pela tabela `ITENS_PROVA_2009.csv`; a extração de enunciado/alternativas ainda requer conferência visual quando `extractionStatus` não é `auto_extracted`.


In [ ]:
manifest={"years":sorted(zip_paths),"source":"INEP official microdata ZIPs",
          "records":len(questions),"needs_review":int(questions.extractionStatus.eq("needs_review").sum()) if len(questions) else 0,
          "created_utc":pd.Timestamp.utcnow().isoformat()}
(OUT/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(manifest,ensure_ascii=False,indent=2))
